# Password Generator — Exploration & Development

This notebook documents selected stages of the development process,
including initial ideas, implementation, debugging, validation,
edge-case testing, and final verification.

## 1. Initial Design

In [ ]:
# """This is the main design"""

# def generate_from_items(items: list, count: int) -> str:
#     pass

# def generate_pin(length: int) -> str:
#     pass

# def generate_random_pass(length: int) -> str:
#     pass

# def generate_memorable_password(words: list, count: int) -> str:
#     pass

## 2. Random Selection & Word Source

In [3]:
# import random
import secrets
import string
from collections.abc import Sequence  # Sequence allows both list and string inputs

import nltk
from nltk.corpus import words


def select_random_items(items: Sequence, count: int) -> list:
    selected_items = [] # an space to keep the random choices

    for _ in range(count): # "_" is used because the iteration value is not needed
        selected_items.append(secrets.choice(items)) # to append the chosen item from the random module and its submodule choice to the selected items space
    return selected_items


result = select_random_items(["A", "B", "C", "D", "E"], 3) # a list and repeating to test the definition
print(result)
result = select_random_items(items=['1', '2', '3', '4', '5', '6', '7', '8', '9'], count=4) # a list and repeating to test the definition
print(result)

['D', 'E', 'C']
['6', '1', '6', '3']


In [4]:
def select_unique_random_items(items: Sequence, count: int) -> list:
    if count > len(items):
        raise ValueError("Not enough unique items available!")

    return secrets.SystemRandom().sample(items, count)

In [5]:
nltk.download("words")

[nltk_data] Downloading package words to /home/neda/nltk_data...
[nltk_data]   Package words is already up-to-date!


True

In [6]:
word_list = words.words()

four_letter_words = [
    word.lower()
    for word in word_list
    if len(word) == 4 and word.isalpha()
]

print(len(four_letter_words))
print(four_letter_words[:30])
print(len(word_list))
print(word_list[:10])

5513
['aani', 'aaru', 'abac', 'abas', 'abba', 'abby', 'abed', 'abel', 'abet', 'abey', 'abie', 'abir', 'able', 'ably', 'abox', 'absi', 'abut', 'acca', 'acer', 'ache', 'achy', 'acid', 'acis', 'acle', 'acme', 'acne', 'acor', 'acre', 'acta', 'acts']
236736
['A', 'a', 'aa', 'aal', 'aalii', 'aam', 'Aani', 'aardvark', 'aardwolf', 'Aaron']


## 3. Password Generators

In [7]:
def generate_rules(password_type):
    if password_type == "pin":
        rules = {
        "number" : True,
        "separator" : "",
        }
        

    elif password_type == "random":
        rules = {
        "number": True,
        "uppercase": True,
        "symbol": True,
        "separator": "",
        }
    elif password_type =="memorable":
        rules = {
            "word" : True,
            "word_length" : 4,
            "separator" : "-",
        }

    else:
        raise ValueError("Invalid password type")

    return rules

result = generate_rules(password_type= "random")
print(result)

{'number': True, 'uppercase': True, 'symbol': True, 'separator': ''}


In [8]:
# The first pass generator --> PIN
def generate_pin(rules: dict, length: int = 4) -> str:
    """Generating the Pin

    :param rules: includes number and separator
    :type rules: dict
    :param length: defaults to 4
    :type length: int"""
    
    if not rules["number"]:
        raise ValueError("PIN must contain numbers!")

    return "".join(select_random_items(string.digits, length))

rules = generate_rules("pin")
result = generate_pin(rules)
print(result)

7018


In [9]:
# The Second pass generator --> Random

symbols = "!@#$%^&*"

def generate_random_password(rules: dict, length: int = 8) -> str:
    """Generating a random password 

    :param rules: includes at least (a number, a uppercase letter, a symbol)
    :type rules: dict
    :param length: defaults to 8
    :type length: int
    :raises ValueError: the password should contain at least 3 items
    :return: items should be in given length (>3) with no space or separator
    :rtype: str
    """
    items = string.digits + string.ascii_letters + symbols

    required_count = 0
    password_chars = []

    if rules["number"]:
        required_count += 1
        password_chars.append(secrets.choice(string.digits))

    if rules["uppercase"]:
        required_count += 1
        password_chars.append(secrets.choice(string.ascii_uppercase))

    if rules["symbol"]:
        required_count += 1
        password_chars.append(secrets.choice(symbols))

    if length < required_count:
        raise ValueError("The length is too low!")

    remaining = length - len(password_chars)

    for _ in range(remaining):
        password_chars.append(secrets.choice(items))
    

    secrets.SystemRandom().shuffle(password_chars)
    return "".join(password_chars)
    # return "".join(select_random_items(items, length))

rules = generate_rules("random")
result = generate_random_password(rules)
print(result)

k^3oesAa


In [ ]:
# The third pass generator --> Memorable
# Memorable words evaluation and word source

rules = generate_rules("memorable")

valid_words = [
    word.lower()
    for word in word_list
    if len(word) == rules["word_length"] and word.isalpha()
]

def generate_memorable_password(rules: dict, count: int) -> str:
    """Generating a memorable password

    :param rules: should contain words with a certain word length and their certain count
    :type rules: dict
    :param count: user choice 
    :type count: int
    :return: words separated with "_" and no space.
    :rtype: str
    """
    selected_words = select_unique_random_items(valid_words, count)


    if not rules["word"]:
        raise ValueError("Password must contain words!")

    return rules["separator"].join(selected_words)


rules = generate_rules("memorable")
result = generate_memorable_password(rules, 4)
print(result)


bija-sign-food-snow


In [11]:
# ❌ این Error نیست که باید برطرفش کنیم.
# اتفاقاً این تست دقیقاً برای ایجاد همین خطا نوشته شده بود.

items = ["book", "cake", "make", "pale", "bake"]

result = select_unique_random_items(items, 5)

print(result)
print(len(result))
print(len(set(result)))

['book', 'pale', 'make', 'bake', 'cake']
5
5


## 4. Password Validation

In [12]:
def validate_password(
    password: str, 
    rules: dict, 
    password_type: str, 
    count: int, 
    length: int
) -> bool:
    """test the passwords validation

    :param password: pin, random or memorable
    :type password: str
    :param rules: depending on password type
    :type rules: dict
    :param password_type: _description_
    :type password_type: str
    :param count: _description_
    :type count: int
    :param length: _description_
    :type length: int
    :return: _description_
    :rtype: bool
    """

    if password_type == "pin":
        if rules["number"] and not all(char in string.digits for char in password):
            return False

        if len(password) != length:
            return False
        
    elif password_type == "random":
        if rules["number"] and not any(char in string.digits for char in password):
            return False
        
        if rules["uppercase"] and not any(char in string.ascii_uppercase for char in password):
            return False

        if rules["symbol"] and not any(char in symbols for char in password):
            return False

        if len(password) != length:
            return False

        if rules["separator"] and rules["separator"] in password:
            return False
        

    elif password_type == "memorable":
        separator = rules["separator"]

        if separator not in password:
            return False

        if password.startswith(separator) or password.endswith(separator):
            return False

        if separator * 2 in password:
            return False

        words = password.split(separator)
    
        if len(words) != count:
            return False
        
        if rules["word"]:
            for word in words:
                if not word.isalpha():
                    return False
               
                if len(word) != rules["word_length"]:
                    return False      

    return True

In [13]:
# test
rules = generate_rules("pin")

password = "5831"

result = validate_password(
    password,
    rules,
    "pin",
    count=0,
    length=4
)

print(result)

True


In [14]:
# test
rules = generate_rules("pin")

password = "58bk1"

result = validate_password(
    password,
    rules,
    "pin",
    count=0,
    length=4
)

print(result)

False


In [15]:
rules = generate_rules("random")

password = "n@P9htos8e"

result = validate_password(
    password,
    rules,
    "random",
    count=0,
    length=10

)

print(result)

True


In [16]:
# test
rules = generate_rules("random")

password = "n@p9hto8e"

result = validate_password(
    password,
    rules,
    "random",
    count=0,
    length=10

)

print(result)

False


In [17]:
# test
rules = generate_rules("memorable")

password = "make-cake-bake"

result = validate_password(
    password,
    rules,
    "memorable",
    count=3,
    length=0

)


print(result)

True


In [18]:
# test
password = "book-cake-make"

result = validate_password(
    password,
    rules,
    "memorable",
    count=3,
    length=0
)

print(result)

True


In [19]:
# test
password = "book-cake-make"

result = validate_password(
    password,
    rules,
    "memorable",
    count=4,
    length=0
)

print(result)

False


In [20]:
rules = generate_rules("memorable")

password = "book-cake-make"
separator = rules["separator"]
words = password.split(separator)

print("rules:", rules)
print("separator:", separator)
print("words:", words)
print("count:", count if 'count' in locals() else "not defined")

print(separator not in password)
print(password.startswith(separator))
print(password.endswith(separator))
print(separator * 2 in password)
print(len(words))
print(rules["word"])

for word in words:
    print(word, word.isalpha(), len(word), rules["word_length"])

rules: {'word': True, 'word_length': 4, 'separator': '-'}
separator: -
words: ['book', 'cake', 'make']
count: 0
False
False
False
False
3
True
book True 4 4
cake True 4 4
make True 4 4


## 5. Retry Mechanism

In [21]:
def generate_retry(
    password_type: str,
    rules: dict,
    length: int = 0, 
    count: int = 0,
    max_attempts: int = 50
):

    if not isinstance(max_attempts, int) or max_attempts <= 0:
        raise ValueError("Max attempts must be a positive integer.")

    if password_type == "pin":
        for _ in range(max_attempts):
            password = generate_pin(rules, length)

            if validate_password(
                password,
                rules,
                "pin",
                count=0,
                length=length
            ):
                return password

            raise ValueError("Failed to generate a valid PIN!")

    elif password_type == "random":
        for _ in range(max_attempts):
            password = generate_random_password(rules, length)

            if validate_password(
                password,
                rules,
                "random",
                count=0,
                length=length
            ):
                return password

            raise ValueError("Failed to generate a valid Random Password!")

    elif password_type == "memorable":

        if not isinstance(count, int) or count <= 0:
            raise ValueError("Count must be a positive integer.")

        for _ in range(max_attempts):
            password = generate_memorable_password(rules, count)

            if validate_password(
                password,
                rules,
                "memorable",
                count=count,
                length=0
            ):
                return password

            raise ValueError("Failed to generate a valid Memorable Password!")

    else:
        raise ValueError("Invalid password type!")

In [22]:
count=5       # باید PASS
count=0       # باید ValueError
count=-1      # باید ValueError
count="hello" # باید ValueError

max_attempts=1       # باید PASS
max_attempts=0       # باید ValueError
max_attempts=-1      # باید ValueError
max_attempts="hello" # باید ValueError

In [23]:
# test
rules = generate_rules("memorable")

result = generate_retry(
    password_type="memorable",
    rules=rules,
    count=5
)

print(result)

skee-onyx-toba-dard-homy


In [24]:
len(valid_words)

5513

In [25]:
# test
rules = generate_rules("pin")

result = generate_retry(
    password_type="pin",
    rules=rules,
    length=4
)

print(result)

1949


In [26]:
# test
rules = generate_rules("random")

result = generate_retry(
    password_type="random",
    rules=rules,
    length=10
)

print(result)

2R1!UyBKcJ


In [27]:
# test
rules = generate_rules("memorable")

result = generate_retry(
    password_type="memorable",
    rules=rules,
    count=5
)

print(result)

zone-pray-ruck-cozy-azon


## 6. Input Handling

In [28]:
length = 0
count = 0


def get_positive_integer(prompt: str) -> int:
    while True:
        try:
            value = int(input(prompt))

            if value > 0:
                return value

            print("Please enter a positive integer.")

        except ValueError:
            print("Please enter a valid integer.")


def get_password_type() -> str:
    while True:
        password_type = input(
            "Enter password type (pin, random, memorable): "
        ).strip().lower()

        if password_type in ["pin", "random", "memorable"]:
            return password_type

        print("Invalid password type. Please try again.")


password_type = get_password_type()

if password_type in ("pin", "random"):

    length = get_positive_integer(
        "Enter your preferred password length: "
    )

    rules = generate_rules(password_type)

    result = generate_retry(
        password_type,
        rules,
        length=length
    )

elif password_type == "memorable":

    count = get_positive_integer(
        "Enter your preferred words count: "
    )

    rules = generate_rules(password_type)

    result = generate_retry(
        password_type,
        rules,
        count=count
    )
else:
    raise ValueError("Invalid password type")

print(result)


54600


## 7. Final Verification

In [29]:
# test
rules = generate_rules("memorable")

result = generate_memorable_password(
    rules,
    count=6
)

print(result)

ilya-here-pian-walt-took-whim


In [30]:
# test
rules = generate_rules("memorable")

result = generate_retry(
    password_type="memorable",
    rules=rules,
    count=0
)

print(result)

ValueError: Count must be a positive integer.

In [ ]:
# test
rules = generate_rules("random")

result = generate_retry(
    password_type="random",
    rules=rules,
    length=12
)

print(result)

g!Ub7MQz%Gll


In [ ]:
# test
rules = generate_rules("pin")

result = generate_retry(
    password_type="pin",
    rules=rules,
    length=6
)

print(result)

193267


In [ ]:
# test
rules = generate_rules("memorable")

result = generate_retry(
    password_type="memorable",
    rules=rules,
    count=5
)

print(result)

moth-lupe-stey-yite-loop
